In [ ]:
import strawberryfields as sf
from strawberryfields.ops import *
import numpy as np
from scipy.special import erfc
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from helper_functions import protocol_noise_free as nf
from scipy.optimize import curve_fit

In [ ]:
from importlib import reload

def reload_protocols():
    reload(nf)

## Theoretical results (Helstrom Bound) CS vs DSS 

In [ ]:
N_helstrom = np.arange(0, 2, 0.02)
b_helstrom = np.arange(0, 1, 0.01)

# Theoretical surface
N_mesh, beta_mesh = np.meshgrid(N_helstrom, b_helstrom, indexing="ij")
p_err_theory = nf.helstrom_bound(N_mesh, beta_mesh)

z_theory = p_err_theory
z_theory_cs = nf.helstrom_bound(N_mesh, 0 * beta_mesh)

# Figure
fig = go.Figure()

# Theory surface
fig.add_trace(go.Surface(x=N_mesh, y=beta_mesh, z=z_theory, colorscale="Reds_r", opacity=1, showscale=False, name="Theory"))

fig.add_trace(go.Surface(x=N_mesh, y=beta_mesh, z=z_theory_cs, colorscale="Blues_r", opacity=1, showscale=False, name="Theory"))

# Layout
fig.update_layout(title="Helstrom Error probability", width=800, height=800, 
                scene=dict(xaxis_title="N", yaxis_title="β", zaxis=dict(title="P_err",type="log"),aspectmode="cube"))

fig.show()

# Coherent States Noise-Free

## Produce Data

In [ ]:
num_samples=5000
alpha_grid = np.linspace(0, np.sqrt(2), 50)
N_cs, p_err_cs = nf.perr_cs(alpha_grid=alpha_grid, homodyne_angle=0, num_samples=num_samples)

np.savez(f"data/perr_data_cs_a{len(alpha_grid)}_S{num_samples}.npz", N_cs=N_cs, alpha_grid=alpha_grid, p_err_cs=p_err_cs)

## Load Data

In [ ]:
data = np.load("data/noise_free/perr_data_cs_a100_S3000.npz")

N_cs = data["N_cs"]
alpha_cs = data["alpha_grid"]
perr_cs = data["p_err_cs"]
beta_cs = np.zeros(len(N_cs))

# DSS Noise-Free

## Produce Data

In [ ]:
N_grid = np.linspace(0, 2, 50)
beta_grid = np.linspace(0, 1, 50)
num_samples=10000
p_err = nf.perr_dss(N_grid=N_grid, beta_grid=beta_grid, homodyne_angle=0, num_samples=num_samples)
np.savez(f"data/perr_data_dss_N{len(N_grid)}_b{len(beta_grid)}_S{num_samples}.npz", N_grid=N_grid, beta_grid=beta_grid, p_err=p_err)

## Load data

In [ ]:
data = np.load("data/noise_free/perr_data_dss_N50_b50_S10000.npz")

N_dss = data["N_grid"]
beta_dss = data["beta_grid"]
perr_dss = data["p_err"]

## DSS/ CS Fit 

In [ ]:
reload_protocols()
params_cs, params_err_cs, params_dss, params_err_dss = nf.fit_homodyne_perr(N_cs, beta_cs, N_dss, beta_dss, perr_cs, perr_dss, dss = True, cs = True, data = False)

## Squeezing Investigation


In [ ]:
reload_protocols()
params_opt, params_opt_err = nf.plot_optimal_squeezing(params_cs, params_dss)

In [ ]:
N_comparison = np.linspace(0, 2, 100)
beta_optimal = nf.beta_opt(N_comparison, *params_opt)
#====================================================
perr_helstrom_dss = nf.helstrom_bound(N_comparison, beta_optimal)
perr_helstrom_cs = nf.helstrom_bound(N_comparison, 0)
#====================================================
perr_cs_low = nf.model_cs((N_comparison, beta_optimal), *(params_cs - params_err_cs))
perr_cs_proper = nf.model_cs((N_comparison, beta_optimal), *params_cs)
perr_cs_high = nf.model_cs((N_comparison, beta_optimal), *(params_cs + params_err_cs))
#====================================================
perr_dss_low = nf.model_dss((N_comparison, beta_optimal), *(params_dss - params_err_dss))
perr_dss_proper = nf.model_dss((N_comparison, beta_optimal), *params_dss)
perr_dss_high = nf.model_dss((N_comparison, beta_optimal), *(params_dss + params_err_dss))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 3), dpi=300)

ax[0].set_ylabel(r'$P_{\text{err}}$')

ax[0].plot(N_comparison, perr_helstrom_cs, linestyle='--', color='k', label = r'$P^{\text{(MIN)}}_{CS}$')
ax[0].plot(N_comparison, perr_cs_proper, label = r'$P_{\text{err}}\left(|\pm a \rangle \right)$', color='b')

ax[1].plot(N_comparison, perr_helstrom_dss, linestyle='--', color='k', label = r'$P^{\text{(MIN)}}_{DSS}$')
ax[1].plot(N_comparison, perr_dss_proper, label = r'$P_{\text{err}}\left(|\pm a, r \rangle \right)$', color='r')

for axis in ax:
    axis.legend()
    axis.set_xlabel(r'$N$')

ax[1].text(0.5, 0.2, s = r'$\beta = \beta_{\text{optimal}}$', fontsize=13)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(4, 3), dpi=200)
plt.xlabel(r'$N$')
plt.ylabel(r'$|P^{\text{(MIN)}} - P_{\text{err}}|$')

plt.plot(N_comparison, abs(perr_helstrom_dss - perr_dss_proper), color='r',
         label = r'|$P^{\text{(MIN)}}_{DSS}$ - $P_{\text{err}}\left(|\pm a, r \rangle \right)$|')

plt.plot(N_comparison, abs(perr_helstrom_cs - perr_cs_proper), color='b',
         label = r'|$P^{\text{(MIN)}}_{CS}$ - $P_{\text{err}}\left(|\pm a \rangle \right)$|')

plt.legend()
plt.tight_layout()
plt.show()